# Notebook 04: Visual CNN Model

Transfer-learning CNN using MobileNetV2 pretrained on ImageNet for 4-class anemia severity classification from conjunctiva images.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

sns.set_style("whitegrid")
BASE_DIR = Path("..")
CLASSES = ["Normal", "Mild", "Moderate", "Severe"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
RANDOM_STATE = 42
print(f"TensorFlow: {tf.__version__}")


## 2. Load Metadata & Build tf.data Pipelines

In [ ]:
df = pd.read_csv(BASE_DIR / "data" / "metadata.csv")

le_label = LabelEncoder()
le_label.fit(CLASSES)
df["label"] = le_label.transform(df["diagnosis"])

df_train, df_temp = train_test_split(
    df, test_size=0.2, stratify=df["diagnosis"], random_state=RANDOM_STATE
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp["diagnosis"], random_state=RANDOM_STATE
)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
], name="augmentation")

def make_dataset(dataframe, augment=False, shuffle=True):
    paths  = [str(BASE_DIR / p) for p in dataframe["image_path"]]
    labels = dataframe["label"].values

    def load_and_preprocess(path, label):
        raw   = tf.io.read_file(path)
        image = tf.image.decode_png(raw, channels=3)
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(dataframe), seed=RANDOM_STATE)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(lambda img, lbl: (augmentation(img, training=True), lbl),
                    num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(df_train, augment=True)
val_ds   = make_dataset(df_val,   augment=False, shuffle=False)
test_ds  = make_dataset(df_test,  augment=False, shuffle=False)
print(f"Train batches: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


## 3. Class Weights

In [ ]:
y_train = df_train["label"].values
class_weights_arr = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights_arr))
print("Class weights:", {CLASSES[k]: round(v, 3) for k, v in class_weight_dict.items()})


## 4. Build MobileNetV2 Transfer Learning Model

In [ ]:
def build_visual_model(num_classes: int = 4, trainable_from_layer: int = 100) -> Model:
    """Build a MobileNetV2-based classifier for anemia severity.

    Architecture:
        - MobileNetV2 backbone (pretrained on ImageNet)
        - GlobalAveragePooling2D
        - Dense(128, relu) + Dropout(0.5)
        - Dense(num_classes, softmax)

    Args:
        num_classes: Number of output classes.
        trainable_from_layer: Layers at index >= this value are unfrozen for fine-tuning.

    Returns:
        Compiled Keras Model.
    """
    base_model = MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    # Phase 1: freeze entire backbone
    base_model.trainable = False

    inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3), name="image_input")
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D(name="gap")(x)
    x       = layers.Dense(128, activation="relu", name="dense_head")(x)
    x       = layers.Dropout(0.5, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = Model(inputs, outputs, name="MobileNetV2_Anemia")
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, base_model

visual_model, base_model = build_visual_model()
visual_model.summary()


## 5. Phase 1: Train Classification Head

In [ ]:
os.makedirs("../models/saved_models", exist_ok=True)

callbacks_phase1 = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ModelCheckpoint(
        "../models/saved_models/visual_model_phase1_best.h5",
        monitor="val_accuracy", save_best_only=True
    ),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
]

history1 = visual_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1
)


## 6. Phase 2: Fine-Tune Last Layers

In [ ]:
# Unfreeze the top layers of MobileNetV2 for fine-tuning
UNFREEZE_FROM = 100
base_model.trainable = True
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False

total_trainable = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable backbone layers (fine-tune phase): {total_trainable}")

visual_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_phase2 = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ModelCheckpoint(
        "../models/saved_models/visual_model_finetuned.h5",
        monitor="val_accuracy", save_best_only=True
    ),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7),
]

history2 = visual_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2
)


## 7. Training History Plots

In [ ]:
def plot_history(h1, h2=None, save_path=None):
    """Plot training and validation accuracy/loss curves."""
    acc1  = h1.history["accuracy"]
    vacc1 = h1.history["val_accuracy"]
    loss1 = h1.history["loss"]
    vloss1= h1.history["val_loss"]

    if h2:
        acc  = acc1  + h2.history["accuracy"]
        vacc = vacc1 + h2.history["val_accuracy"]
        loss = loss1 + h2.history["loss"]
        vloss= vloss1+ h2.history["val_loss"]
        split_epoch = len(acc1)
    else:
        acc, vacc, loss, vloss = acc1, vacc1, loss1, vloss1
        split_epoch = None

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(acc) + 1)

    axes[0].plot(epochs, acc,  label="Train Accuracy")
    axes[0].plot(epochs, vacc, label="Val Accuracy")
    if split_epoch:
        axes[0].axvline(split_epoch, color="grey", linestyle="--", label="Fine-tune start")
    axes[0].set_title("Model Accuracy"); axes[0].legend()

    axes[1].plot(epochs, loss,  label="Train Loss")
    axes[1].plot(epochs, vloss, label="Val Loss")
    if split_epoch:
        axes[1].axvline(split_epoch, color="grey", linestyle="--", label="Fine-tune start")
    axes[1].set_title("Model Loss"); axes[1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

plot_history(history1, history2,
             save_path="../models/saved_models/visual_training_history.png")


## 8. Evaluate on Test Set

In [ ]:
y_true, y_pred_all = [], []
for images, labels in test_ds:
    preds = visual_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred_all.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred_all)

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
print(f"Test Accuracy : {acc:.4f}")
print(f"Test Macro F1 : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Visual CNN — Confusion Matrix (Test Set)", fontweight="bold")
plt.tight_layout()
plt.savefig("../models/saved_models/visual_model_confusion_matrix.png", bbox_inches="tight")
plt.show()

# Critical metric: recall for Severe anemia
severe_idx = le_label.transform(["Severe"])[0]
severe_recall = (y_true[y_true == severe_idx] == y_pred[y_true == severe_idx]).mean()
print(f"\nSevere Anemia Recall: {severe_recall:.4f}  (critical safety metric)")


## 9. 5-Fold Cross-Validation

In [ ]:
all_images = []
all_labels = []

for imgs, lbls in tf.data.Dataset.from_tensor_slices(
    ([str(BASE_DIR / p) for p in df["image_path"]], df["label"].values)
).map(
    lambda p, l: (
        tf.cast(tf.image.resize(tf.image.decode_png(tf.io.read_file(p), channels=3), IMG_SIZE), tf.float32) / 255.0,
        l
    )
).batch(BATCH_SIZE).prefetch(AUTOTUNE):
    all_images.append(imgs.numpy())
    all_labels.append(lbls.numpy())

X_img = np.concatenate(all_images, axis=0)
y_all = np.concatenate(all_labels, axis=0)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_acc, cv_f1 = [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_img, y_all), 1):
    fold_model, _ = build_visual_model()
    fold_ds_train = (
        tf.data.Dataset.from_tensor_slices((X_img[train_idx], y_all[train_idx]))
        .shuffle(len(train_idx), seed=fold).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    )
    fold_ds_val = (
        tf.data.Dataset.from_tensor_slices((X_img[val_idx], y_all[val_idx]))
        .batch(BATCH_SIZE).prefetch(AUTOTUNE)
    )
    fold_model.fit(fold_ds_train, epochs=15, verbose=0,
                   class_weight=class_weight_dict)
    y_v = y_all[val_idx]
    y_p = np.argmax(fold_model.predict(fold_ds_val, verbose=0), axis=1)
    cv_acc.append(accuracy_score(y_v, y_p))
    cv_f1.append(f1_score(y_v, y_p, average="macro", zero_division=0))
    print(f"Fold {fold}: Acc={cv_acc[-1]:.4f}  F1={cv_f1[-1]:.4f}")

print(f"\nCV Accuracy : {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"CV Macro F1 : {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")
